# GPU RAID — воркер на Google Colab (только платный тариф)

⚠️ **Только Colab Pro / Pay-As-You-Go с положительным балансом compute units.**
FAQ бесплатного Colab прямо запрещает «running distributed computing workers»;
на платных тарифах это ограничение снято. Этот ноутбук откажется работать,
пока вы не подтвердите платный тариф в конфиге.

Рекомендуемый GPU: **L4 (24 ГБ)** — тянет fp8-видео (Wan) и Flux.

In [ ]:
# ================= КОНФИГ =================
REPO_URL = "https://github.com/Weloyo/ComfyUI-GPU-RAID"
TOKEN = ""                 # пусто = сгенерировать
MODEL_PRESET = "sdxl"      # "sdxl" | "minimax_h3" | "none"
USE_DRIVE_CACHE = True      # кэш моделей (в т.ч. hf_preset) в Google Drive — не перекачивать каждый рантайм
I_USE_PAID_COLAB = False   # <-- поставьте True, подтверждая Pro/PAYG

# Автоподключение к мастеру (gist-rendezvous): задайте GIST_ID тот же, что в
# панели GPU RAID (Режимы → rendezvous), а токен GitHub положите в Colab
# Secrets (ключ 🔑 слева, имя GH_TOKEN, fine-grained токен с правом только на
# gists). Тогда строки копировать не нужно — мастер подхватит воркера сам,
# а режимы Эко/«Сразу гасить» смогут останавливать этот рантайм автоматически.
GIST_ID = ""
MAX_SESSION_MIN = 0        # самостраховка: погасить рантайм через N минут (0 = выкл)

# minimax_h3 (~40 ГБ весов): на стандартной Colab RAM тесно — если есть Runtime →
# Change runtime type → High-RAM, включите его. --cache-none экономит RAM всегда.
EXTRA_ARGS = ("--cache-none",) if MODEL_PRESET == "minimax_h3" else ()  # L4/A100: bf16/fp8 работают, force-fp16 не нужен

HF_TOKEN = ""
GH_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "")
except Exception:
    pass
try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or (userdata.get("GH_TOKEN") or "")
except Exception:
    pass
assert I_USE_PAID_COLAB, ("Подтвердите платный Colab: I_USE_PAID_COLAB = True. "
                          "На бесплатном тарифе распределённые воркеры запрещены правилами.")
if GIST_ID and not GH_TOKEN:
    print("! GIST_ID задан, но секрета GH_TOKEN нет — автоподключение работать не будет")
print("config ok")

In [ ]:
# ============ ИСХОДНИКИ РАСШИРЕНИЯ ============
import os, sys, subprocess
SRC = "/content/gpu-raid-src"
if not os.path.isdir(SRC):
    assert subprocess.run(["git", "clone", "--depth", "1", REPO_URL, SRC]).returncode == 0, \
        "git clone не удался — проверьте REPO_URL"
sys.path.insert(0, os.path.join(SRC, "scripts"))
import worker_bootstrap as wb
TOKEN = wb.gen_token(TOKEN)
print("TOKEN:", TOKEN)
subprocess.run(["nvidia-smi", "-L"])

In [ ]:
# ============ (опция) КЭШ МОДЕЛЕЙ В DRIVE ============
COMFY_DIR = "/content/ComfyUI"
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = "/content/drive/MyDrive/gpuraid_models"
    os.makedirs(cache, exist_ok=True)
    # Отдельная плоская папка под hf_preset (напр. minimax_h3): просто копии
    # готовых файлов, БЕЗ вложенных symlink-цепочек huggingface_hub — Drive
    # FUSE-mount ненадёжен именно для них (проверено: крупные файлы по 5-20 ГБ
    # тихо "терялись" при кэшировании через HF_HOME=Drive). Первая сессия
    # качает с HF и сохраняет сюда копию, все следующие копируют отсюда вместо
    # повторного скачивания.
    hf_drive_cache = "/content/drive/MyDrive/gpuraid_hf_cache"
    os.makedirs(hf_drive_cache, exist_ok=True)
    # после установки ComfyUI (следующая ячейка) файлы из кэша прилинкуются:
    def link_drive_models():
        n = 0
        for folder in os.listdir(cache):
            src_dir = os.path.join(cache, folder)
            if not os.path.isdir(src_dir):
                continue
            for f in os.listdir(src_dir):
                n += wb._link(os.path.join(src_dir, f), COMFY_DIR, folder, f)
        print(f"[drive] прилинковано: {n}")
else:
    hf_drive_cache = None
    def link_drive_models():
        pass
print("ok")

In [ ]:
# ============ УСТАНОВКА + ЗАПУСК + ТУННЕЛЬ ============
wb.install_comfy(COMFY_DIR)
link_drive_models()
info = wb.bring_up(
    gpuraid_src=SRC,
    comfy_dir=COMFY_DIR,
    token=TOKEN,
    gpus=(0,),
    base_port=8188,
    extra_args=EXTRA_ARGS,
    use_datasets=False,
    hf_preset=MODEL_PRESET,
    hf_token=HF_TOKEN,
    name_prefix="colab",
    drive_cache_dir=hf_drive_cache,
    gist_id=GIST_ID,
    gh_token=GH_TOKEN,
    max_session_min=MAX_SESSION_MIN,
)

In [ ]:
# ============ WATCHDOG ============
# Следит за процессами и туннелем (умерший туннель перезапустит и перепубликует
# адрес в gist). Завершается сам, когда мастер пришлёт команду остановки
# (режимы Эко/«Сразу гасить») — тогда рантайм освобождается runtime.unassign().
wb.watchdog(info)